<a href="https://colab.research.google.com/github/jackapolinarisa-25/DAM-APOLINARISA2501010333/blob/main/UTS_DAM_2501010333.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import time
import hashlib
import requests
import pandas as pd
from datetime import datetime
from urllib.robotparser import RobotFileParser

# Susun URL repositori Anda secara otomatis agar tidak terpotong
user_gh = "jackapolinarisa-25"
repo_gh = "UTS-DATA-ACQUISITION_2501010333"
url_gh = "https://github.com/jackapolinarisa-25/UTS-DATA-ACQUISITION_2501010333"

!rm -rf {repo_gh}
!git clone {url_gh}

# Mengatur lokasi folder kerja
base_dir = repo_gh
raw_dir = os.path.join(base_dir, "data/raw")
proses_dir = os.path.join(base_dir, "data/proses")
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(proses_dir, exist_ok=True)

HEADERS = {
    "User-Agent": "MahasiswaINSTIKI-UTS-DAM/1.0 (2501010333@instiki.ac.id)"
}

def hitung_md5(filepath):
    hash_md5 = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

provenance_records = []
print("Cek isi folder data/raw:", os.listdir(raw_dir))

Cloning into 'UTS-DATA-ACQUISITION_2501010333'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 25 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 292.13 KiB | 5.84 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Cek isi folder data/raw: ['bali_tourist_destination.csv', '.gitkeep']


In [11]:

# Menentukan path langsung ke file CSV Anda
raw_csv_path = os.path.join(raw_dir, "bali_tourist_destination.csv")
print(f"Membaca file: {raw_csv_path}")

# Membaca dataset dengan parameter yang tepat
df_wisata = pd.read_csv(raw_csv_path, sep=",", na_values=["", "NA", "null", "-"])

# Identifikasi jumlah baris, kolom, dan ukuran file
baris, kolom = df_wisata.shape
ukuran_kb = os.path.getsize(raw_csv_path) / 1024
estimasi_1_tahun_kb = (ukuran_kb * 12) * 1.30  # Ukuran x 12 bulan + cadangan 30%

print(f"\nJenis Data: Terstruktur (Tabel CSV)")
print(f"Jumlah Baris: {baris} baris | Jumlah Kolom: {kolom} kolom")
print(f"Ukuran Berkas Saat Ini: {ukuran_kb:.2f} KB")
print(f"Estimasi Kebutuhan Storage 1 Tahun (+30% cadangan): {estimasi_1_tahun_kb:.2f} KB")
print("\nTipe Data Setiap Kolom:")
print(df_wisata.dtypes)

# Simpan versi bersih ke folder data/proses
df_wisata.to_csv(os.path.join(proses_dir, "wisata_bali_bersih.csv"), index=False)

# Catat provenance Sumber 1
provenance_records.append({
    "Sumber": "Dataset Tempat Wisata Bali (CSV)",
    "URL": "https://github.com",
    "Tanggal_Akses": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "Lisensi": "Open Data / CC-BY",
    "MD5_Hash": hitung_md5(raw_csv_path)
})
print("\nKotak Kode 2 Selesai!")

Membaca file: UTS-DATA-ACQUISITION_2501010333/data/raw/bali_tourist_destination.csv

Jenis Data: Terstruktur (Tabel CSV)
Jumlah Baris: 1452 baris | Jumlah Kolom: 11 kolom
Ukuran Berkas Saat Ini: 707.37 KB
Estimasi Kebutuhan Storage 1 Tahun (+30% cadangan): 11035.03 KB

Tipe Data Setiap Kolom:
id_tempat              object
nama_tempat_wisata     object
kategori               object
kecamatan              object
kabupaten_kota         object
rating                float64
jumlah_rating           int64
latitude              float64
longitude             float64
link_google_maps       object
link_gambar            object
dtype: object

Kotak Kode 2 Selesai!


In [16]:

api_url = "https://open-meteo.com"
params = {
    "latitude": -8.65,
    "longitude": 115.2167,
    "start_date": "2025-01-01",
    "end_date": "2025-02-15",  # 46 baris (> minimal 30 baris)
    "daily": "temperature_2m_mean,precipitation_sum",
    "timezone": "Asia/Makassar"
}

for attempt in range(3):
    try:
        resp_api = requests.get(api_url, params=params, headers=HEADERS, timeout=10)
        if resp_api.status_code == 200:
            raw_json_path = os.path.join(raw_dir, "cuaca_bali_raw.json")
            with open(raw_json_path, "w", encoding="utf-8") as f:
                f.write(resp_api.text)
            data_json = resp_api.json()
            print("Status 200 OK: Data API berhasil disimpan ke data/raw/cuaca_bali_raw.json!")
            break
    except Exception as e:
        print(f"Percobaan {attempt+1} gagal: {e}")
    time.sleep(2)

# Normalisasi JSON menjadi tabel DataFrame
df_cuaca = pd.DataFrame({
    "tanggal": data_json["daily"]["time"],
    "suhu_rata_rata_c": data_json["daily"]["temperature_2m_mean"],
    "curah_hujan_mm": data_json["daily"]["precipitation_sum"]
})

print(f"Total baris tabel API setelah normalisasi: {len(df_cuaca)} baris")
display(df_cuaca.head(30))

# Simpan versi bersih ke data/proses
df_cuaca.to_json(os.path.join(proses_dir, "cuaca_bali_bersih.json"), orient="records", indent=2)

# Catat provenance Sumber 2
provenance_records.append({
    "Sumber": "Open-Meteo Historical Weather API",
    "URL": resp_api.url,
    "Tanggal_Akses": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "Lisensi": "CC BY 4.0 (Open-Meteo)",
    "MD5_Hash": hitung_md5(raw_json_path)
})
print("Kotak Kode 3 Selesai!")

Percobaan 1 gagal: Expecting value: line 1 column 1 (char 0)
Percobaan 2 gagal: Expecting value: line 1 column 1 (char 0)
Percobaan 3 gagal: Expecting value: line 1 column 1 (char 0)
Total baris tabel API setelah normalisasi: 46 baris


,tanggal,suhu_rata_rata_c,curah_hujan_mm
0,2025-01-01,25.7,10.6
1,2025-01-02,24.8,29.4
2,2025-01-03,24.9,15.8
3,2025-01-04,24.4,15.9
4,2025-01-05,25.0,13.4
5,2025-01-06,25.5,26.9
6,2025-01-07,26.2,18.6
7,2025-01-08,26.5,10.0
8,2025-01-09,25.9,17.9
9,2025-01-10,25.3,8.8


Kotak Kode 3 Selesai!


In [19]:
target_url = "https://id.wikipedia.org/wiki/Daftar_kabupaten_dan_kota_di_Bali"
robots_url = "https://id.wikipedia.org/wiki/robots.txt"

# 1. Unduh robots.txt menggunakan User-Agent resmi kampus Anda agar tidak diblokir
resp_robots = requests.get(robots_url, headers=HEADERS, timeout=10)
rp = RobotFileParser()
rp.parse(resp_robots.text.splitlines())

boleh_scrape = rp.can_fetch(HEADERS["User-Agent"], target_url)
print(f"Bukti Pemeriksaan robots.txt -> Diizinkan: {boleh_scrape}")

if boleh_scrape:
    time.sleep(2)  # Jeda sopan antar-request
    resp_web = requests.get(target_url, headers=HEADERS, timeout=10)
    resp_web.raise_for_status()

    # Simpan HTML mentah tanpa diubah ke data/raw
    raw_html_path = os.path.join(raw_dir, "wilayah_bali_raw.html")
    with open(raw_html_path, "w", encoding="utf-8") as f:
        f.write(resp_web.text)

    # Ekstrak tabel kabupaten/kota dari Wikipedia
    tabel_wiki = pd.read_html(resp_web.text)
    # Mengambil tabel pertama yang berisi daftar kabupaten dan kota
    df_wilayah = tabel_wiki[0].copy()

    # Simpan versi proses
    df_wilayah.to_html(os.path.join(proses_dir, "wilayah_bali_bersih.html"), index=False)
    print("Berhasil mengambil tabel wilayah Bali dari Wikipedia:")
    display(df_wilayah.head())

    # Catat provenance Sumber 3
    provenance_records.append({
        "Sumber": "Wikipedia Kabupaten dan Kota di Bali",
        "URL": target_url,
        "Tanggal_Akses": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Lisensi": "CC BY-SA 4.0",
        "MD5_Hash": hitung_md5(raw_html_path)
    })
print("Kotak Kode 4 Selesai!")

Bukti Pemeriksaan robots.txt -> Diizinkan: True
Berhasil mengambil tabel wilayah Bali dari Wikipedia:


,No.,Kabupaten/kota,Ibu kota,Bupati/wali kota,Luas wilayah (km2)[1],Jumlah penduduk (2025)[1],Kecamatan,Kelurahan/desa,Lambang,Peta lokasi
0,1,Kabupaten Badung,Mangupura,I Wayan Adi Arnawa,397395,537.739,6,16/46,NaN,NaN
1,2,Kabupaten Bangli,Bangli,Sang Nyoman Sedana Arta,526764,260.526,4,4/68,NaN,NaN
2,3,Kabupaten Buleleng,Singaraja,I Nyoman Sutjidra,"1.322,124",828.156,9,19/129,NaN,NaN
3,4,Kabupaten Gianyar,Gianyar,I Made Agus Mahayastra,363782,504.845,7,6/64,NaN,NaN
4,5,Kabupaten Jembrana,Negara,I Made Kembang Hartawan,847515,330.221,5,10/41,NaN,NaN
5,6,Kabupaten Karangasem,Amlapura,I Gusti Putu Parwata,838899,538.890,8,3/75,NaN,NaN
6,7,Kabupaten Klungkung,Semarapura,I Made Satria,312115,223.723,4,6/53,NaN,NaN
7,8,Kabupaten Tabanan,Singasana,I Komang Gede Sanjaya,848539,478.393,10,-/133,NaN,NaN
8,9,Kota Denpasar,-,I Gusti Ngurah Jaya Negara,125694,673.270,4,16/27,NaN,NaN


Kotak Kode 4 Selesai!


In [20]:

# 1. Mencari kolom nama Kabupaten/Kota pada tabel Wikipedia secara otomatis
kolom_nama_wiki = [col for col in df_wilayah.columns if "Kabupaten" in str(col) or "kota" in str(col).lower()][0]
df_wilayah_clean = df_wilayah.copy()

# 2. Membersihkan format teks kunci wilayah agar cocok persis (misal: "Kota Denpasar", "Kabupaten Badung")
df_wilayah_clean["kunci_wilayah"] = (
    df_wilayah_clean[kolom_nama_wiki]
    .astype(str)
    .str.replace(r"\[.*?\]", "", regex=True)
    .str.strip()
)
df_wilayah_clean = df_wilayah_clean.drop_duplicates(subset=["kunci_wilayah"])

# 3. Menggabungkan dataset wisata (CSV) dengan profil wilayah (Scraping)
df_gabungan = pd.merge(
    df_wisata,
    df_wilayah_clean,
    left_on="kabupaten_kota",
    right_on="kunci_wilayah",
    how="left",
    validate="many_to_one"
)

# 4. Pembuktian tidak terjadi duplikasi baris akibat proses merge
assert len(df_gabungan) == len(df_wisata), "Peringatan: Terjadi duplikasi baris!"
print(f"Validasi Merge Sukses! Jumlah baris tetap {len(df_gabungan)} baris tanpa duplikasi.")
display(df_gabungan[["id_tempat", "nama_tempat_wisata", "kategori", "kabupaten_kota"]].head())

# 5. Simpan hasil gabungan dan catatan provenance ke folder proyek
df_gabungan.to_csv(os.path.join(proses_dir, "data_gabungan_final.csv"), index=False)
df_provenance = pd.DataFrame(provenance_records)
df_provenance.to_csv(os.path.join(base_dir, "catatan_provenance.csv"), index=False)

print("\n=== CATATAN PROVENANCE LENGKAP (URL, TANGGAL, LISENSI, MD5) ===")
display(df_provenance)

Validasi Merge Sukses! Jumlah baris tetap 1452 baris tanpa duplikasi.


,id_tempat,nama_tempat_wisata,kategori,kabupaten_kota
0,BL0101001,Patung Titi Banda,Umum,Kota Denpasar
1,BL0101002,Uma.palak.(parkir.2),Rekreasi,Kota Denpasar
2,BL0101003,Tukad Bindu Park,Rekreasi,Kota Denpasar
3,BL0101004,Pantai Padang Galak,Alam,Kota Denpasar
4,BL0101005,Museum Le Mayeur,Budaya,Kota Denpasar



=== CATATAN PROVENANCE LENGKAP (URL, TANGGAL, LISENSI, MD5) ===


,Sumber,URL,Tanggal_Akses,Lisensi,MD5_Hash
0,Dataset Tempat Wisata Bali (CSV),https://github.com,2026-11-25 18:20:26,Open Data / CC-BY,cc969fcea82e75d16436354feb74d6d7
1,Open-Meteo Historical Weather API,https://archive-api.open-meteo.com/v1/archive?...,2026-11-25 18:23:30,CC BY 4.0 (Open-Meteo),8a54dd6ee2a5b4090568605a060d3468
2,Open-Meteo Historical Weather API,https://archive-api.open-meteo.com/v1/archive?...,2026-11-25 18:29:00,CC BY 4.0 (Open-Meteo),48a5ce7e7d26b3256fe61c5eab283bd5
3,Open-Meteo Historical Weather API,https://open-meteo.com/?latitude=-8.65&longitu...,2026-11-25 18:37:17,CC BY 4.0 (Open-Meteo),0f20b69198d1da6eccb56e10f945445d
4,Wikipedia Kabupaten dan Kota di Bali,https://id.wikipedia.org/wiki/Daftar_kabupaten...,2026-11-25 18:38:49,CC BY-SA 4.0,61a2069fd3c783b8fcded18548958fbd
5,Wikipedia Kabupaten dan Kota di Bali,https://id.wikipedia.org/wiki/Daftar_kabupaten...,2026-11-25 18:41:16,CC BY-SA 4.0,61a2069fd3c783b8fcded18548958fbd


# Laporan UTS Data Acquisition & Management (Kelas F)
* **Nama:** [Apolinarisa Jack Herment]
* **NIM:** 2501010333
* **Program Studi:** Informatika — INSTIKI
* **Dosen Pengampu:** Dr. I Gede Totok Suryawan, S.Kom., M.T.

---

## 1. Identifikasi Dataset & Asumsi Estimasi Storage (Soal 1)
* **Jenis Data:** Data terstruktur dalam format tabel (`.csv`, `.json`, dan tabel HTML).
* **Cakupan Wilayah & Waktu:** Seluruh Provinsi Bali (9 Kabupaten/Kota) dengan data cuaca historis periode Januari–Februari 2025.
* **Asumsi Perhitungan Storage 1 Tahun:** Diasumsikan penambahan data baru terjadi secara rutin setiap bulan dengan volume ukuran berkas yang konstan (`ukuran berkas × 12 bulan`), ditambah cadangan ruang (*buffer*) sebesar **30%** untuk pencadangan (*backup*), log pemrosesan, dan penyimpanan file hasil pembersihan/penggabungan.

---

## 2. Identifikasi PII / Quasi-Identifier & Data Ethics Checklist (Soal 2)
* **Identifikasi PII & Quasi-Identifier:** Dataset destinasi wisata Bali yang digunakan **tidak mengandung *Personally Identifiable Information* (PII)** maupun *quasi-identifier* individu, karena seluruh atribut hanya merepresentasikan informasi entitas tempat umum (nama destinasi, kategori, kecamatan, kabupaten/kota, koordinat geografis, dan rating agregat).

### Data Ethics Checklist (9 Butir)
1. **Kejelasan Tujuan:** Data dikumpulkan semata-mata untuk kepentingan akademik (UTS Data Acquisition & Management) dan analisis sektor pariwisata Bali.
2. **Kepatuhan Lisensi & Ketentuan Sumber:** Seluruh data diambil dari sumber terbuka (Open Data / CC-BY / CC BY-SA 4.0) dan tidak melanggar larangan pengikisan situs komersial (OTA).
3. **Perlindungan Privasi (PII):** Dataset telah diperiksa dan dipastikan bebas dari data pribadi sensitif maupun identitas individu.
4. **Kepatuhan `robots.txt`:** Proses *web scraping* pada Wikipedia telah diverifikasi terlebih dahulu melalui `RobotFileParser` dan dinyatakan diizinkan (`True`).
5. **Identitas *User-Agent* Transparan:** Setiap permintaan HTTP menyertakan `User-Agent` resmi yang mencantumkan identitas akademik (`2501010333@instiki.ac.id`).
6. **Pengendalian Laju Permintaan (*Rate Limiting*):** Setiap request menerapkan `timeout` dan jeda waktu (`time.sleep`) agar tidak membebani server penyedia data.
7. **Integritas Data Mentah (*Raw Data*):** Seluruh data asli dari tiga sumber disimpan utuh di folder `data/raw` tanpa modifikasi dan dilengkapi tanda tangan digital **MD5 Hash**.
8. **Transparansi *Provenance*:** Seluruh sumber data dicatat secara lengkap meliputi URL sumber, waktu akses, lisensi, dan nilai MD5 Hash pada `catatan_provenance.csv`.
9. **Pencegahan Duplikasi & Bias Penggabungan:** Proses integrasi data menggunakan parameter validasi (`validate="many_to_one"`) untuk menjamin keakuratan relasi wilayah tanpa duplikasi buatan.